# 02 - Test Isolation Forest Model

**Elyra Pipeline Node 2** — Load the trained model, run inference on test data, and report evaluation metrics.

**Inputs** (from pipeline or env):
- `MODEL_PATH`: Path to trained model pickle
- `TEST_DATA_PATH`: Path to test features CSV
- `FEATURE_COLS_PATH`: JSON with feature column names
- `THRESHOLD_PATH`: Calibrated threshold JSON

**Outputs**:
- `TEST_REPORT_PATH`: JSON and HTML test report

In [ ]:
import os
import json
from pathlib import Path

MODEL_PATH = os.getenv("MODEL_PATH", "artifacts/model.pkl")
TEST_DATA_PATH = os.getenv("TEST_DATA_PATH", "test_data.csv")
FEATURE_COLS_PATH = os.getenv("FEATURE_COLS_PATH", "artifacts/feature_cols.json")
THRESHOLD_PATH = os.getenv("THRESHOLD_PATH", "artifacts/threshold.json")
TEST_REPORT_PATH = os.getenv("TEST_REPORT_PATH", "artifacts/test_report.json")

In [ ]:
import pandas as pd
import numpy as np
from joblib import load

# Load model
model = load(MODEL_PATH)
with open(FEATURE_COLS_PATH) as f:
    meta = json.load(f)
feature_cols = meta["feature_cols"]
with open(THRESHOLD_PATH) as f:
    thresh_meta = json.load(f)
threshold = thresh_meta["threshold"]

print(f"Model loaded from {MODEL_PATH}")
print(f"Threshold: {threshold:.4f}")

In [ ]:
# Load test data
if Path(TEST_DATA_PATH).exists():
    df = pd.read_csv(TEST_DATA_PATH)
    print(f"Loaded {len(df)} test rows from {TEST_DATA_PATH}")
else:
    # Generate sample test data if not present
    print("No test data found. Generating sample data...")
    np.random.seed(123)
    n_normal, n_anomaly = 80, 20
    X_normal = np.random.exponential(scale=5, size=(n_normal, len(feature_cols))).clip(0, 100)
    X_anomaly = np.random.exponential(scale=50, size=(n_anomaly, len(feature_cols))).clip(50, 500)
    X = np.vstack([X_normal, X_anomaly])
    df = pd.DataFrame(X, columns=feature_cols)
    df["root_cause"] = ["none"] * n_normal + ["fault"] * n_anomaly
    df.to_csv(TEST_DATA_PATH, index=False)
    print(f"Saved sample test data to {TEST_DATA_PATH}")

# Ensure feature columns exist
avail = [c for c in feature_cols if c in df.columns]
if len(avail) < len(feature_cols):
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0
X_test = df[feature_cols].fillna(0).to_numpy()
y_true = (df["root_cause"] != "none").astype(int).to_numpy() if "root_cause" in df.columns else None

In [ ]:
# Run inference
try:
    scores = model.predict_scores(X_test)
except AttributeError:
    scores = -model.decision_function(X_test)

scores = np.asarray(scores, dtype=float)
preds = (scores > threshold).astype(int)

print(f"Scored {len(scores)} samples")
print(f"Predicted anomalies: {preds.sum()}")

In [ ]:
# Compute metrics (when ground truth available)
report = {
    "model_path": str(MODEL_PATH),
    "test_samples": int(len(scores)),
    "predicted_anomalies": int(preds.sum()),
    "threshold": float(threshold),
    "score_mean": float(np.mean(scores)),
    "score_std": float(np.std(scores)),
}

if y_true is not None and len(np.unique(y_true)) >= 2:
    from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
    report["precision"] = float(precision_score(y_true, preds, zero_division=0))
    report["recall"] = float(recall_score(y_true, preds, zero_division=0))
    report["f1"] = float(f1_score(y_true, preds, zero_division=0))
    report["roc_auc"] = float(roc_auc_score(y_true, scores))
    report["pr_auc"] = float(average_precision_score(y_true, scores))
    print("Test Results:")
    for k, v in report.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
else:
    print("No ground truth labels - reporting score statistics only.")
    for k, v in report.items():
        print(f"  {k}: {v}")

In [ ]:
# Save report
Path(TEST_REPORT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(TEST_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

# Optional: save HTML summary
html_path = TEST_REPORT_PATH.replace(".json", ".html")
with open(html_path, "w") as f:
    f.write("<h2>Isolation Forest Test Report</h2><pre>")
    f.write(json.dumps(report, indent=2))
    f.write("</pre>")

print(f"Report saved → {TEST_REPORT_PATH}")